($H_0$): The mean difference between `surface_pressure` and `surface_pressure_roll6` is the same for days with heavy rain (rain_class = 2) and days with no/light rain (rain_class < 2).
($H_0: \mu_{\text{diff\_heavy}} = \mu_{\text{diff\_light}}$)

($H_1$): The mean difference between `surface_pressure` and `surface_pressure_roll6` is significantly different (specifically lower/more negative) during heavy rain events.
($H_1: \mu_{\text{diff\_heavy}} \neq \mu_{\text{diff\_light}}$)

Assuming confidence interval = 0.95

In [9]:
import pandas as pd
from scipy import stats
import numpy as np

In [4]:
df = pd.read_csv("./phnom_penh_weather_processed.csv")

In [11]:
# 0. Setting confidence_interval
confidence_interval  = 0.95

# 1. Create the Delta column
df['pressure_delta'] = df['surface_pressure'] - df['surface_pressure_roll6']

# 2. Split groups
group_a = df[df['rain_class'] < 2]['pressure_delta']
group_b = df[df['rain_class'] == 2]['pressure_delta']

# 3. Calculate statistics for both groups
n1, n2 = len(group_a), len(group_b)
m1, m2 = group_a.mean(), group_b.mean()
v1, v2 = group_a.var(ddof=1), group_b.var(ddof=1)

# 4. Calculate the Difference in Means
diff = m1 - m2

# 5. Calculate Standard Error of the difference
pooled_se = np.sqrt(v1/n1 + v2/n2)

# 6. Calculate Degrees of Freedom (Welch-Satterthwaite)
dof = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))

# 7. Calculate Confidence Interval
t_crit = stats.t.ppf(confidence_interval, dof)
lower = diff - t_crit * pooled_se
upper = diff + t_crit * pooled_se

print(f"Mean Difference (A - B): {diff:.4f}")
print(f"{confidence_interval*100}% Confidence Interval: [{lower:.4f}, {upper:.4f}]")

# 8. Decision Logic
if lower > 0 or upper < 0:
    print("Result: 0 is not in the interval. The difference is statistically significant.")
else:
    print("Result: 0 is in the interval. We cannot prove a significant difference.")

Mean Difference (A - B): 0.0015
95.0% Confidence Interval: [0.0002, 0.0028]
Result: 0 is not in the interval. The difference is statistically significant.
